In [ ]:
# Method 1 - Faster R-CNN (frozen FPN backbone, grid features) + GRU + BAN  [PathVQA]
# Changes 1,2,4,6,8 + CHANGE A: data-scaling experiment.
# CHANGE A = subsample official TRAIN IMAGES at 20/40/60/80/100%, >=3 seeds/cell,
#   fixed val/test, plot Exact Match vs training-fraction with mean+/-std band.
#   Method 1 has a FIXED CNN encoder -> one curve; overlay a ViT curve from the
#   Method 3 (timm) notebook via VIT_SCALING_CSV. COMPUTE: 5x3=15 trainings by default.

# Method 1 - Faster R-CNN (frozen FPN backbone, grid features) + GRU + BAN  [PathVQA]
# Changes 1,2,4,6 + Change 8: native BAN attention overlay (plot_attention_10).
# NOTE: visual encoder is the FROZEN FPN backbone used as a grid, NOT true region proposals.

# ============================================================
# PATHVQA WITH PROPER BAN (Bilinear Attention Networks)
# Following paper [20]: GRU + Faster R-CNN + BAN + Low-rank approx
# CHANGE 1: OFFICIAL PathVQA split (train/validation/test as provided)
# ============================================================

!pip -q install datasets timm nltk rouge-score scikit-learn

# =========================
# IMPORTS
# =========================
import os, re, gc, math, time, json, random, string, hashlib, warnings
import numpy as np, pandas as pd, matplotlib.pyplot as plt

from io import BytesIO
from PIL import Image
from tqdm.auto import tqdm
from collections import Counter

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import torchvision.transforms as T
import torchvision.transforms.functional as TF
from torchvision.models.detection import fasterrcnn_resnet50_fpn

from datasets import load_dataset
from sklearn.metrics import accuracy_score
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from rouge_score import rouge_scorer

import nltk
nltk.download("stopwords", quiet=True)
from nltk.corpus import stopwords

warnings.filterwarnings("ignore")

# =========================
# SEED / DEVICE
# =========================
SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE:", DEVICE)

# =========================
# CONFIG
# =========================
DATASET_NAME = "flaviagiammarino/path-vqa"

IMAGE_SIZE = 224
BATCH_SIZE = 4
NUM_WORKERS = 0
EPOCHS = 15   # CHANGE 4: equalized across all three methods
LR = 1e-4
WEIGHT_DECAY = 1e-4

MAX_Q_LEN = 30
WORD_VOCAB_SIZE = 2200   # CHANGE 4: equalized

EMBED_DIM = 300
GRU_HIDDEN = 1024
DROPOUT = 0.5

# BAN specific parameters
NUM_GLIMPSES = 8
BAN_HIDDEN = 1024
LOW_RANK = 512

OUTPUT_DIR = "./outputs_ban_proper"
os.makedirs(OUTPUT_DIR, exist_ok=True)

STOP_WORDS = set(stopwords.words("english"))
SMOOTH_FN = SmoothingFunction().method1
ROUGE_SCORER = rouge_scorer.RougeScorer(["rouge1","rougeL"], use_stemmer=True)

# =========================
# TEXT NORMALIZATION
# =========================
# CHANGE 2: no stopword removal (it corrupted answers: "no"->unk, "not present"->"present").
def _basic_clean(x):
    x = str(x).lower().strip()
    x = x.translate(str.maketrans("", "", string.punctuation))
    x = re.sub(r"\s+", " ", x)
    return x.strip()

def normalize_question(x, unk="<unk>"):
    x = _basic_clean(x)
    return x if x else unk

def normalize_answer(x, unk="<unk>"):
    x = _basic_clean(x)
    return x if x else unk

# CHANGE 6: derive question type from raw question + raw answer
_YESNO_LEAD = {"is","are","does","do","did","was","were","can","could","has","have","will","should","would","may","am"}
def get_question_type(question_raw, answer_raw):
    a = str(answer_raw).strip().lower()
    if a in ("yes","no"):
        return "yes/no"
    q = _basic_clean(question_raw)
    first = q.split()[0] if q.split() else ""
    if first in ("what","which","whose","name"): return "what"
    if first == "where": return "where"
    if first == "how": return "how"
    if first == "when": return "when"
    if first in _YESNO_LEAD: return "yes/no"
    return "other"

def tokenize(x):
    t = x.split()
    return t if t else ["<unk>"]

# =========================
# IMAGE HANDLING
# =========================
def image_to_pil(obj):
    if isinstance(obj, Image.Image):
        return obj.convert("RGB")
    if isinstance(obj, dict) and obj.get("bytes"):
        return Image.open(BytesIO(obj["bytes"])).convert("RGB")
    if isinstance(obj, dict) and obj.get("path"):
        return Image.open(obj["path"]).convert("RGB")
    return Image.new('RGB', (224, 224), 'black')

def stable_image_id(img_obj, fallback="unknown"):
    try:
        if isinstance(img_obj, dict):
            if img_obj.get("path", None) is not None:
                return str(img_obj["path"])
            if img_obj.get("bytes", None) is not None:
                return hashlib.md5(img_obj["bytes"]).hexdigest()
        if isinstance(img_obj, Image.Image):
            img_rgb = img_obj.convert("RGB")
            buf = BytesIO()
            img_rgb.save(buf, format="PNG")
            return hashlib.md5(buf.getvalue()).hexdigest()
    except Exception:
        return str(fallback)
    return str(fallback)

# =========================
# LOAD DATA
# =========================
print("Loading dataset...")
dataset = load_dataset(DATASET_NAME)

# Get column names
all_split_names = list(dataset.keys())
base_cols = dataset[all_split_names[0]].column_names

def find_col(cols, candidates):
    for c in candidates:
        if c in cols:
            return c
    return None

IMAGE_COL = find_col(base_cols, ["image", "img"])
QUESTION_COL = find_col(base_cols, ["question", "query"])
ANSWER_COL = find_col(base_cols, ["answer", "answers"])
IMAGE_ID_COL = find_col(base_cols, ["image_id", "img_id", "id"])

print(f"IMAGE_COL: {IMAGE_COL}, QUESTION_COL: {QUESTION_COL}, ANSWER_COL: {ANSWER_COL}")

# Build full dataframe with proper image IDs
rows = []
for s in dataset:
    print(f"Processing {s} split...")
    for i in tqdm(range(len(dataset[s])), desc=f"Loading {s}"):
        try:
            ex = dataset[s][i]
            img_obj = ex[IMAGE_COL]
            
            # Get stable image ID
            if IMAGE_ID_COL and IMAGE_ID_COL in ex:
                image_id = str(ex[IMAGE_ID_COL])
            else:
                image_id = stable_image_id(img_obj, fallback=f"{s}_{i}")
            
            rows.append({
                "split": s,
                "idx": i,
                "image_id": image_id,
                "q_raw": ex.get(QUESTION_COL, ""),
                "a_raw": ex.get(ANSWER_COL, ""),
                "q": normalize_question(ex.get(QUESTION_COL, "")),
                "a": normalize_answer(ex.get(ANSWER_COL, "")),
                "q_type": get_question_type(ex.get(QUESTION_COL, ""), ex.get(ANSWER_COL, ""))
            })
        except Exception as e:
            print(f"Error processing {s}_{i}: {e}")
            continue

df = pd.DataFrame(rows)
print(f"Total samples: {len(df)}")
print(f"Unique images: {df['image_id'].nunique()}")

# =========================
# CHANGE 1: OFFICIAL SPLIT (each row already carries its true split label)
# =========================
train_df_full = df[df.split == "train"].reset_index(drop=True)   # full official training set
val_df   = df[df.split == "validation"].reset_index(drop=True)
test_df  = df[df.split == "test"].reset_index(drop=True)

print("\n===== OFFICIAL PATHVQA SPLIT =====")
print("Train images:", train_df_full["image_id"].nunique(), " Train QA:", len(train_df_full))
print("Val images  :", val_df["image_id"].nunique(),   " Val QA  :", len(val_df))
print("Test images :", test_df["image_id"].nunique(),  " Test QA :", len(test_df))

# =========================
# VOCABULARY BUILDING
# =========================
# CHANGE A: vocab + answer space are built ONCE from the FULL training set and held
# FIXED across every fraction, so the ONLY variable in the sweep is the AMOUNT of
# training data (not the label space).
wc = Counter()
for t in train_df_full.q.tolist() + train_df_full.a.tolist():
    wc.update(tokenize(t))

itos = ["<pad>", "<unk>"] + [w for w, _ in wc.most_common(WORD_VOCAB_SIZE - 2)]
stoi = {w: i for i, w in enumerate(itos)}
PAD = stoi["<pad>"]
UNK = stoi["<unk>"]

print(f"Vocabulary size: {len(stoi)}")

def encode_q(x):
    t = tokenize(x)[:MAX_Q_LEN]
    ids = [stoi.get(w, UNK) for w in t]
    ids = ids + [PAD] * (MAX_Q_LEN - len(ids))
    return ids, min(len(t), MAX_Q_LEN)

# =========================
# ANSWER SPACE
# =========================
ans_list = sorted(set(train_df_full.a.tolist()))
if "<unk>" not in ans_list:
    ans_list.append("<unk>")
a2i = {a: i for i, a in enumerate(ans_list)}
i2a = {i: a for a, i in a2i.items()}

print(f"Answer vocabulary size: {len(ans_list)}")

# =========================
# DATASET CLASS
# =========================
tf = T.Compose([T.Resize((224, 224)), T.ToTensor(),
                T.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])])  # CHANGE 4

class VQADataset(Dataset):
    def __init__(self, df):
        self.df = df
        
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, i):
        r = self.df.iloc[i]
        ex = dataset[r.split][r.idx]
        img = tf(image_to_pil(ex.get(IMAGE_COL)))
        q, ql = encode_q(r.q)
        return {
            "img": img,
            "q": torch.tensor(q, dtype=torch.long),
            "ql": torch.tensor(ql, dtype=torch.long),
            "y": torch.tensor(a2i.get(r.a, a2i["<unk>"]), dtype=torch.long),
            "gt": r.a,
            "q_raw": r.q_raw,
            "image_id": r.image_id,
            "q_type": r.q_type   # CHANGE 6
        }

# val/test loaders are FIXED for every run in the scaling sweep
val_dl = DataLoader(VQADataset(val_df), batch_size=BATCH_SIZE, num_workers=NUM_WORKERS)
test_dl = DataLoader(VQADataset(test_df), batch_size=BATCH_SIZE, num_workers=NUM_WORKERS)

# =========================
# FASTER R-CNN FEATURE EXTRACTOR
# =========================
class FasterRCNNFeatureExtractor(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = fasterrcnn_resnet50_fpn(pretrained=True)
        self.backbone = self.model.backbone
        self.rpn = self.model.rpn
        
        for param in self.backbone.parameters():
            param.requires_grad = False
        
        self.out_channels = 256
        
    def forward(self, x):
        features = self.backbone(x)
        feature_map = list(features.values())[0]
        return feature_map

# =========================
# PROPER BAN IMPLEMENTATION
# =========================
class BilinearAttention(nn.Module):
    def __init__(self, v_dim, q_dim, num_glimpses, hidden_size, low_rank):
        super().__init__()
        self.num_glimpses = num_glimpses
        self.low_rank = low_rank
        
        self.U = nn.Linear(v_dim, low_rank, bias=False)
        self.V = nn.Linear(q_dim, low_rank, bias=False)
        
        self.h = nn.Parameter(torch.Tensor(low_rank, num_glimpses))
        nn.init.xavier_uniform_(self.h)
        
        self.proj = nn.Linear(v_dim * num_glimpses, hidden_size)
        self.dropout = nn.Dropout(0.5)
        
    def forward(self, v, q, return_attention=False):
        u = self.U(v)
        v_proj = self.V(q).unsqueeze(1)
        bilinear = u * v_proj
        attention = torch.softmax(bilinear @ self.h, dim=1)
        attended = torch.einsum('bng,bnd->bgd', attention, v)
        attended = attended.reshape(attended.size(0), -1)
        output = self.dropout(self.proj(attended))
        
        if return_attention:
            return output, attention
        return output

class GRUEncoder(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_layers=1, dropout=0.5):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.gru = nn.GRU(embed_dim, hidden_dim, num_layers, 
                         batch_first=True, bidirectional=False, dropout=dropout)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, x, lengths):
        x = self.dropout(self.embedding(x))
        packed = nn.utils.rnn.pack_padded_sequence(x, lengths.cpu(), 
                                                   batch_first=True, enforce_sorted=False)
        _, hidden = self.gru(packed)
        hidden = hidden[-1]
        return self.dropout(hidden)

class BANModel(nn.Module):
    def __init__(self, vocab_size, num_answers, embed_dim=300, gru_dim=1024, 
                 num_glimpses=8, ban_hidden=1024, low_rank=512):
        super().__init__()
        
        self.image_encoder = FasterRCNNFeatureExtractor()
        self.image_proj = nn.Linear(self.image_encoder.out_channels, ban_hidden)
        
        self.question_encoder = GRUEncoder(vocab_size, embed_dim, gru_dim, dropout=DROPOUT)
        self.question_proj = nn.Linear(gru_dim, ban_hidden)
        
        self.ban = BilinearAttention(
            v_dim=ban_hidden,
            q_dim=ban_hidden,
            num_glimpses=num_glimpses,
            hidden_size=ban_hidden,
            low_rank=low_rank
        )
        
        self.classifier = nn.Sequential(
            nn.Linear(ban_hidden, 1024),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(1024, num_answers)
        )
        
    def forward(self, images, questions, lengths, return_attention=False):
        v = self.image_encoder(images)
        B, C, H, W = v.shape
        v = v.view(B, C, H*W).permute(0, 2, 1)
        v = self.image_proj(v)
        
        q = self.question_encoder(questions, lengths)
        q = self.question_proj(q)
        
        if return_attention:
            fused, attention = self.ban(v, q, return_attention=True)
            att_map = attention.mean(dim=2).view(B, H, W)
            logits = self.classifier(fused)
            return logits, att_map
        else:
            fused = self.ban(v, q, return_attention=False)
            logits = self.classifier(fused)
            return logits

# =========================
# MODEL BUILDER + CHANGE A HELPERS
# =========================
def build_model_and_opt():
    m = BANModel(
        vocab_size=len(stoi), num_answers=len(a2i), embed_dim=EMBED_DIM,
        gru_dim=GRU_HIDDEN, num_glimpses=NUM_GLIMPSES,
        ban_hidden=BAN_HIDDEN, low_rank=LOW_RANK
    ).to(DEVICE)
    o = torch.optim.Adam(m.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    return m, o

def seed_everything(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

# CHANGE A: subsample by IMAGE (carry all QAs for chosen images) -> no image leakage
def subsample_train_by_image(full_df, frac, seed):
    imgs = sorted(full_df["image_id"].unique().tolist())
    rng = random.Random(seed); rng.shuffle(imgs)
    k = max(1, int(round(frac * len(imgs))))
    keep = set(imgs[:k])
    return full_df[full_df["image_id"].isin(keep)].reset_index(drop=True)

# module-level handles used by train_epoch()/evaluate() (rebound each sweep run)
model, optimizer, train_dl = None, None, None

# =========================
# METRICS FUNCTIONS
# =========================
def compute_bleu(pred, gold):
    pred_tokens = pred.split()
    gold_tokens = gold.split()
    try:
        bleu1 = sentence_bleu([gold_tokens], pred_tokens, weights=(1,0,0,0), smoothing_function=SMOOTH_FN) * 100
        bleu2 = sentence_bleu([gold_tokens], pred_tokens, weights=(0.5,0.5,0,0), smoothing_function=SMOOTH_FN) * 100
        bleu3 = sentence_bleu([gold_tokens], pred_tokens, weights=(1/3,1/3,1/3,0), smoothing_function=SMOOTH_FN) * 100
        return bleu1, bleu2, bleu3
    except:
        return 0, 0, 0

def compute_rouge(pred, gold):
    try:
        scores = ROUGE_SCORER.score(gold, pred)
        return scores['rouge1'].fmeasure * 100, scores['rougeL'].fmeasure * 100
    except:
        return 0, 0

def compute_f1(pred, gold):
    pred_tokens = set(pred.split())
    gold_tokens = set(gold.split())
    if not pred_tokens or not gold_tokens:
        return 0
    common = pred_tokens & gold_tokens
    precision = len(common) / len(pred_tokens)
    recall = len(common) / len(gold_tokens)
    return 2 * precision * recall / (precision + recall) * 100 if (precision + recall) > 0 else 0

# =========================
# TRAINING AND EVALUATION
# =========================
def train_epoch():
    model.train()
    total_loss = 0
    for batch in tqdm(train_dl, desc="Training"):
        img = batch["img"].to(DEVICE)
        q = batch["q"].to(DEVICE)
        ql = batch["ql"].to(DEVICE)
        y = batch["y"].to(DEVICE)
        
        logits = model(img, q, ql)
        loss = F.cross_entropy(logits, y)
        
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        
        total_loss += loss.item()
    return total_loss / len(train_dl)

@torch.no_grad()
def evaluate(dl, save_path=None):
    model.eval()
    golds, preds, qtypes = [], [], []   # CHANGE 6
    rows = []
    
    for batch in tqdm(dl, desc="Evaluating"):
        img = batch["img"].to(DEVICE)
        q = batch["q"].to(DEVICE)
        ql = batch["ql"].to(DEVICE)
        
        logits = model(img, q, ql)
        pred_ids = torch.argmax(logits, dim=1).cpu().tolist()
        
        for i, pid in enumerate(pred_ids):
            pred = i2a[pid]
            gold = batch["gt"][i]
            preds.append(pred)
            golds.append(gold)
            qtypes.append(batch["q_type"][i])   # CHANGE 6
            rows.append({
                "image_id": batch["image_id"][i],
                "question": batch["q_raw"][i],
                "gold": gold,
                "pred": pred,
                "correct": gold == pred
            })
    
    if save_path:
        pd.DataFrame(rows).to_csv(save_path, index=False)
    
    # Compute metrics
    yesno_idx = [i for i, g in enumerate(golds) if g in ['yes', 'no']]
    open_idx = [i for i, g in enumerate(golds) if g not in ['yes', 'no']]
    
    metrics = {
        'overall_exact_match_pct': np.mean([g == p for g, p in zip(golds, preds)]) * 100 if golds else 0,
        'yes_no_accuracy_pct': 0,
        'open_exact_match_pct': 0,
        'macro_f1_pct': 0,
        'bleu1_pct': 0,
        'bleu2_pct': 0,
        'bleu3_pct': 0,
        'rouge1_f_pct': 0,
        'rougeL_f_pct': 0,
        'total_samples': len(golds)
    }
    
    if yesno_idx:
        yesno_golds = [golds[i] for i in yesno_idx]
        yesno_preds = [preds[i] for i in yesno_idx]
        metrics['yes_no_accuracy_pct'] = accuracy_score(yesno_golds, yesno_preds) * 100
    
    if open_idx:
        open_golds = [golds[i] for i in open_idx]
        open_preds = [preds[i] for i in open_idx]
        
        metrics['open_exact_match_pct'] = np.mean([g == p for g, p in zip(open_golds, open_preds)]) * 100
        metrics['macro_f1_pct'] = np.mean([compute_f1(p, g) for g, p in zip(open_golds, open_preds)])
        
        bleu1, bleu2, bleu3 = zip(*[compute_bleu(p, g) for g, p in zip(open_golds, open_preds)])
        metrics['bleu1_pct'] = np.mean(bleu1)
        metrics['bleu2_pct'] = np.mean(bleu2)
        metrics['bleu3_pct'] = np.mean(bleu3)
        
        rouge1, rougeL = zip(*[compute_rouge(p, g) for g, p in zip(open_golds, open_preds)])
        metrics['rouge1_f_pct'] = np.mean(rouge1)
        metrics['rougeL_f_pct'] = np.mean(rougeL)
    
    # CHANGE 6: per-question-type exact match
    order = ["yes/no","what","where","how","when","other"]
    buckets = {}
    for g, pr, t in zip(golds, preds, qtypes):
        buckets.setdefault(t, []).append(int(g == pr))
    per_type = {}
    for t in order:
        if t in buckets and buckets[t]:
            per_type[t] = (float(np.mean(buckets[t])*100), len(buckets[t]))
    for t, hits in buckets.items():
        if t not in per_type:
            per_type[t] = (float(np.mean(hits)*100), len(hits))
    metrics['per_type'] = per_type
    
    return metrics, rows

# ============================================================
# CHANGE A: DATA-SCALING EXPERIMENT
# ------------------------------------------------------------
# Subsample OFFICIAL TRAINING IMAGES (not QA pairs) at each fraction, carry ALL
# QAs for the chosen images; val/test stay FIXED; >=3 seeds per cell.
#
# NOTE ON "CNN vs ViT" (from the Change A spec): Method 1's visual encoder is a
# FIXED CNN (frozen Faster R-CNN backbone). It cannot be swapped to a ViT here,
# so THIS notebook produces the CNN curve only. To get the two-curve CNN-vs-ViT
# figure, run the SAME sweep in the encoder-swappable notebook (Method 3 / timm)
# with a ViT backbone, save its scaling_results.csv, and set VIT_SCALING_CSV below.
#
# !!! COMPUTE WARNING: len(FRACTIONS) x len(SEEDS) full trainings. With the
# defaults that is 5 x 3 = 15 runs of SCALING_EPOCHS each. On a T4 with the BAN
# model this is LONG. For a quick smoke test set e.g. SEEDS=[42],
# FRACTIONS=[0.2, 1.0], SCALING_EPOCHS=3. Results save incrementally and the
# sweep RESUMES from scaling_results.csv, so it is safe to stop/restart.
# ============================================================
FRACTIONS = [0.2, 0.4, 0.6, 0.8, 1.0]
SEEDS = [42, 1, 2]
SCALING_EPOCHS = EPOCHS   # keep identical across all fractions; lower to cut compute

scaling_csv = f"{OUTPUT_DIR}/scaling_results.csv"
if os.path.exists(scaling_csv):
    scaling_rows = pd.read_csv(scaling_csv).to_dict("records")
    done = {(round(float(r["fraction"]),3), int(r["seed"])) for r in scaling_rows}
    print(f"Resuming scaling sweep: {len(done)} cells already complete.")
else:
    scaling_rows, done = [], set()

canon_metrics, canon_rows = None, None
for frac in FRACTIONS:
    for seed in SEEDS:
        if (round(frac,3), int(seed)) in done:
            print(f"skip  frac={frac:.0%} seed={seed} (already done)")
            continue
        print("\n" + "#"*70)
        print(f"# SCALING RUN | fraction={frac:.0%} | seed={seed} | epochs={SCALING_EPOCHS}")
        print("#"*70)

        seed_everything(seed)
        sub_train_df = subsample_train_by_image(train_df_full, frac, seed)
        print(f"train images={sub_train_df['image_id'].nunique()} | train QA={len(sub_train_df)}")

        train_dl = DataLoader(VQADataset(sub_train_df), batch_size=BATCH_SIZE,
                              shuffle=True, num_workers=NUM_WORKERS)
        model, optimizer = build_model_and_opt()

        for epoch in range(SCALING_EPOCHS):
            tl = train_epoch()

        tm, tr = evaluate(test_dl)   # fixed test set; no per-cell CSV
        row = {
            "fraction": float(frac), "seed": int(seed),
            "train_images": int(sub_train_df["image_id"].nunique()),
            "train_qa": int(len(sub_train_df)),
            "overall_em": tm["overall_exact_match_pct"],
            "yes_no_acc": tm["yes_no_accuracy_pct"],
            "open_em": tm["open_exact_match_pct"],
            "macro_f1": tm["macro_f1_pct"],
        }
        scaling_rows.append(row)
        pd.DataFrame(scaling_rows).to_csv(scaling_csv, index=False)   # incremental save
        print(f"-> overall EM={row['overall_em']:.2f}% | open EM={row['open_em']:.2f}% | yes/no={row['yes_no_acc']:.2f}%")

        # capture the full-data run as the canonical Method-1 result + viz model
        if abs(frac - 1.0) < 1e-9 and seed == SEEDS[-1]:
            canon_metrics, canon_rows = tm, tr

        gc.collect()
        if DEVICE == "cuda":
            torch.cuda.empty_cache()

# fallback: if the canonical (100%, last seed) cell was skipped via resume,
# train one full-data model so the report + attention overlay still work.
if canon_metrics is None:
    print("\nCanonical 100% cell was resumed/skipped -> training one full-data model for reporting.")
    seed_everything(SEEDS[-1])
    train_dl = DataLoader(VQADataset(train_df_full), batch_size=BATCH_SIZE,
                          shuffle=True, num_workers=NUM_WORKERS)
    model, optimizer = build_model_and_opt()
    for epoch in range(SCALING_EPOCHS):
        train_epoch()
    canon_metrics, canon_rows = evaluate(test_dl)

# ---- aggregate mean +/- std per fraction ----
res = pd.DataFrame(scaling_rows)
agg = (res.groupby("fraction")["overall_em"]
          .agg(["mean","std","count"]).reset_index().sort_values("fraction"))
agg["std"] = agg["std"].fillna(0.0)
print("\n===== SCALING CURVE (Overall EM, mean +/- std over seeds) =====")
print(f"{'Fraction':>10}{'Mean EM':>12}{'Std':>10}{'Seeds':>8}")
for _, r in agg.iterrows():
    print(f"{r['fraction']:>10.0%}{r['mean']:>12.2f}{r['std']:>10.2f}{int(r['count']):>8}")

# ---- plot the CNN scaling curve with shaded mean +/- std band ----
x = (agg["fraction"] * 100).values
mean = agg["mean"].values; std = agg["std"].values
plt.figure(figsize=(7,5))
plt.plot(x, mean, marker="o", label="Faster R-CNN + GRU + BAN (CNN encoder)")
plt.fill_between(x, mean - std, mean + std, alpha=0.2)
plt.xlabel("Training data (% of official PathVQA train images)")
plt.ylabel("Overall Exact Match (%)")
plt.title(f"Data-scaling on PathVQA (Method 1, CNN)\nmean +/- std over seeds {SEEDS}")
plt.grid(True, alpha=0.3); plt.legend(); plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/scaling_curve.png", dpi=150); plt.show()

# ---- OPTIONAL: overlay a ViT curve from the encoder-swappable (Method 3) notebook ----
VIT_SCALING_CSV = None   # e.g. "./outputs_method3_vit/scaling_results.csv"
if VIT_SCALING_CSV and os.path.exists(VIT_SCALING_CSV):
    vit = pd.read_csv(VIT_SCALING_CSV)
    vagg = (vit.groupby("fraction")["overall_em"].agg(["mean","std"])
               .reset_index().sort_values("fraction"))
    vagg["std"] = vagg["std"].fillna(0.0)
    vx = (vagg["fraction"] * 100).values
    plt.figure(figsize=(7,5))
    plt.plot(x, mean, marker="o", label="CNN (this notebook)")
    plt.fill_between(x, mean - std, mean + std, alpha=0.2)
    plt.plot(vx, vagg["mean"].values, marker="s", label="ViT (Method 3 notebook)")
    plt.fill_between(vx, vagg["mean"].values - vagg["std"].values,
                     vagg["mean"].values + vagg["std"].values, alpha=0.2)
    plt.xlabel("Training data (%)"); plt.ylabel("Overall Exact Match (%)")
    plt.title("CNN vs ViT data-scaling on PathVQA")
    plt.grid(True, alpha=0.3); plt.legend(); plt.tight_layout()
    plt.savefig(f"{OUTPUT_DIR}/scaling_curve_cnn_vs_vit.png", dpi=150); plt.show()

print("\nLIMIT: PathVQA caps at ~2,499 training images, so this curve characterizes")
print("behavior WITHIN PathVQA's data regime; it cannot show a CNN/ViT crossover.")

# =========================
# CANONICAL FULL-DATA (100%) TEST REPORT
# =========================
print("\n" + "="*60)
print("FINAL TEST RESULTS (100% training data)")
print("="*60)
test_metrics, test_rows = canon_metrics, canon_rows
pd.DataFrame(test_rows).to_csv(f"{OUTPUT_DIR}/test_predictions.csv", index=False)

# Print comprehensive results
print("\n" + "="*100)
print("TEST METRICS (Following BAN Paper [20])")
print("="*100)
print(f"{'Metric':<30} {'Score':<10}")
print("-"*40)
print(f"{'Overall Exact Match':<30} {test_metrics['overall_exact_match_pct']:>6.2f}%")
print(f"{'Yes/No Accuracy':<30} {test_metrics['yes_no_accuracy_pct']:>6.2f}%")
print(f"{'Open-ended Exact Match':<30} {test_metrics['open_exact_match_pct']:>6.2f}%")
print(f"{'Macro F1 Score':<30} {test_metrics['macro_f1_pct']:>6.2f}%")
print(f"{'BLEU-1':<30} {test_metrics['bleu1_pct']:>6.2f}%")
print(f"{'BLEU-2':<30} {test_metrics['bleu2_pct']:>6.2f}%")
print(f"{'BLEU-3':<30} {test_metrics['bleu3_pct']:>6.2f}%")
print(f"{'ROUGE-1':<30} {test_metrics['rouge1_f_pct']:>6.2f}%")
print(f"{'ROUGE-L':<30} {test_metrics['rougeL_f_pct']:>6.2f}%")

# CHANGE 6: per-question-type exact match
print("\n" + "-"*60)
print("PER-QUESTION-TYPE EXACT MATCH")
print("-"*60)
print(f"{'Question Type':<14}{'Exact Match (%)':>18}{'N':>10}")
for qtype, (acc, n) in test_metrics.get('per_type', {}).items():
    print(f"{qtype:<14}{acc:>18.2f}{n:>10}")

# Print 10 sample predictions
# =========================
# CHANGE 8: ATTENTION OVERLAY (native BAN attention) for Method 1
# Method 1 already exposes a spatial attention map via return_attention=True
# (shape B x H x W from the bilinear attention), so we visualize that directly.
# =========================
@torch.no_grad()
def plot_attention_10(model, df_for_viz, model_name="BAN", num_samples=10):
    model.eval()
    ds = VQADataset(df_for_viz)
    idxs = list(range(len(ds)))
    random.Random(SEED).shuffle(idxs)
    idxs = idxs[:min(num_samples, len(idxs))]
    for k, idx in enumerate(idxs, 1):
        item = ds[idx]
        img = item["img"].unsqueeze(0).to(DEVICE)
        q   = item["q"].unsqueeze(0).to(DEVICE)
        ql  = item["ql"].unsqueeze(0).to(DEVICE)

        logits, att_map = model(img, q, ql, return_attention=True)  # att_map: (1,H,W)
        pred = i2a.get(int(torch.argmax(logits, dim=1).item()), "<missing>")
        gold = item["gt"]

        # de-normalize image for display
        img_np = item["img"].permute(1, 2, 0).cpu().numpy()
        mean = np.array([0.485, 0.456, 0.406]); std = np.array([0.229, 0.224, 0.225])
        img_np = np.clip(std * img_np + mean, 0, 1)

        att = att_map[0].detach().cpu().numpy().astype(np.float32)
        att = (att - att.min()) / (att.max() - att.min() + 1e-8)

        print("=" * 100)
        print(f"Sample {k} | Q-type: {item['q_type']}")
        print("Question :", item["q_raw"])
        print("GT       :", repr(gold), "| Pred:", repr(pred),
              "|", "CORRECT" if gold == pred else "WRONG")

        plt.figure(figsize=(11, 5))
        plt.subplot(1, 2, 1); plt.imshow(img_np); plt.axis("off"); plt.title("Image")
        plt.subplot(1, 2, 2); plt.imshow(img_np)
        plt.imshow(att, cmap="jet", alpha=0.5,
                   extent=(0, img_np.shape[1], img_np.shape[0], 0))
        plt.axis("off"); plt.title("BAN Attention Overlay")
        plt.suptitle(f"{model_name}\nQ: {item['q_raw']}\nGT: {gold} | Pred: {pred}", fontsize=10)
        plt.tight_layout(); plt.show()

# Produce the attention figures for Method 1 (uses the canonical full-data model)
if model is not None:
    plot_attention_10(model, test_df, model_name="Faster R-CNN + GRU + BAN", num_samples=10)

print("\n" + "="*60)
print("10 SAMPLE TEST PREDICTIONS")
print("="*60)

for i, row in enumerate(test_rows[:10]):
    print(f"\n{i+1}. Image ID: {row['image_id']}")
    print(f"   Question: {row['question'][:100]}...")
    print(f"   Ground Truth: {row['gold']}")
    print(f"   Prediction: {row['pred']}")
    print(f"   Status: {'✓ CORRECT' if row['correct'] else '✗ WRONG'}")

# Save all results
pd.DataFrame([test_metrics]).to_csv(f"{OUTPUT_DIR}/test_metrics.csv", index=False)
pd.DataFrame(test_rows).to_csv(f"{OUTPUT_DIR}/test_predictions.csv", index=False)

print(f"\n✅ Results saved to {OUTPUT_DIR}")
print(f"   - test_metrics.csv: All metrics")
print(f"   - test_predictions.csv: All predictions with image IDs")
print("\n✅ Experiment Complete!")
